# Notebook 2 - Data Analysis with pandas

**AFDP 2026 - Python Workshop (Block 2, about 45 minutes)**

In Notebook 1 you typed every number in by hand. Real actuarial work starts with a file - a policy extract, a claims listing, an experience study - and the Python library for working with tables of data is **pandas**. In this notebook you will load a health-insurance dataset, explore it, filter and group it, build a claim-frequency table by age band, look at the severity distribution, deal with missing values, and draw charts with **matplotlib** and **seaborn**.

**The dataset.** `us_health_insurance_dataset_afdp.csv` has 1,338 US policyholders with their age, sex, BMI, number of children, smoking status, region and annual medical charges. It is small enough to understand completely, which is exactly what you want when learning. (Source: [Kaggle - US Health Insurance Dataset](https://www.kaggle.com/datasets/teertha/ushealthinsurancedataset).)

## 1. Importing the libraries

Libraries are collections of ready-made functions. The four below are the everyday toolkit for data work, and the short aliases (`pd`, `np`, `plt`, `sns`) are universal conventions - every example you find online uses them.

In [ ]:
import numpy as np                  # numerical arrays and maths
import pandas as pd                 # tables of data (DataFrames)
import matplotlib.pyplot as plt     # charts
import seaborn as sns               # nicer statistical charts, built on matplotlib

print("pandas version:", pd.__version__)

## 2. Loading the dataset

The cell below is written so that it works wherever you run it. In Google Colab it will open a file-chooser the first time; pick `us_health_insurance_dataset_afdp.csv` from your computer. (You can also drag the file into the folder icon on the left of the Colab screen before running the cell.) On your own laptop it simply reads the file from the same folder as the notebook.

*Note for Colab users: uploaded files disappear when the session ends, so you may need to upload again tomorrow. Hosting the file at a web address and pasting it into `DATA_URL` avoids that.*

In [ ]:
import os
import pandas as pd

FILE_NAME = "us_health_insurance_dataset_afdp.csv"
DATA_URL = "https://raw.githubusercontent.com/rohanyashraj/afdp-python-training/main/2026%20AFDP%20Python%20Training/us_health_insurance_dataset_afdp.csv"   # Direct link to the CSV in the course GitHub repo; set to "" to upload the file manually instead

if DATA_URL:
    insurance_data = pd.read_csv(DATA_URL)
elif os.path.exists(FILE_NAME):
    insurance_data = pd.read_csv(FILE_NAME)
else:
    try:
        from google.colab import files          # only exists inside Google Colab
        print("Please choose", FILE_NAME, "from your computer in the dialog below.")
        files.upload()
        insurance_data = pd.read_csv(FILE_NAME)
    except ImportError:
        raise FileNotFoundError(f"Put {FILE_NAME} in the same folder as this notebook and run this cell again.")

print("Loaded", len(insurance_data), "rows and", insurance_data.shape[1], "columns.")

## 3. First look at the data

Before analysing anything, look at it. These four commands are the first thing an experienced analyst runs on any new file.

In [ ]:
insurance_data.head()        # first five rows

In [ ]:
insurance_data.info()        # column names, types, and whether anything is missing

In [ ]:
insurance_data.describe()    # summary statistics for the numeric columns

In [ ]:
# Categorical columns: what values do they take, and how many of each?
print(insurance_data["smoker"].value_counts())
print()
print(insurance_data["region"].value_counts())

**Things to notice**

- `charges` is heavily skewed: the mean (about 13,270) is much higher than the median (about 9,382). That is typical of claims data and we will come back to it.
- `info()` shows no missing values. Real data is rarely this kind.
- `object` (or `str`) in the `Dtype` column means text.

## 4. Selecting and filtering

A DataFrame is a table. Square brackets pick out columns; a condition inside square brackets picks out rows. This is the pandas equivalent of a filter in Excel, except it is repeatable and you can chain conditions together.

In [ ]:
# One column -> a Series; a list of columns -> a smaller DataFrame
ages = insurance_data["age"]
subset = insurance_data[["age", "bmi", "smoker", "charges"]]
subset.head()

In [ ]:
# Rows where a condition is true
smokers = insurance_data[insurance_data["smoker"] == "yes"]
print("Number of smokers:", len(smokers))

# Two conditions: use & for 'and', | for 'or', and brackets around each condition
high_risk = insurance_data[(insurance_data["smoker"] == "yes") & (insurance_data["bmi"] > 30)]
print("Smokers with BMI over 30:", len(high_risk))
print(f"Their average charges: {high_risk['charges'].mean():,.0f}")
print(f"Overall average charges (all policyholders): {insurance_data['charges'].mean():,.0f}")

In [ ]:
# Sorting: the ten most expensive policyholders
insurance_data.sort_values("charges", ascending=False).head(10)

### A quick word on NumPy

pandas is built on top of NumPy, which handles the fast arithmetic. You will mostly use pandas directly, but NumPy functions are useful for percentiles and for maths on whole columns at once. Notice how the results agree with `describe()` above.

In [ ]:
charges = insurance_data["charges"].to_numpy()

print(f"Mean:        {np.mean(charges):,.2f}")
print(f"Median:      {np.median(charges):,.2f}")
print(f"95th pctile: {np.percentile(charges, 95):,.2f}")
print(f"Std dev:     {np.std(charges):,.2f}")

# Arithmetic on a whole column at once - no loop needed
insurance_data["charges_thousands"] = insurance_data["charges"] / 1000
insurance_data[["charges", "charges_thousands"]].head(3)

## 5. Actuarial example: claim frequency by age band

This dataset records annual medical charges rather than individual claims, so let us define a **large claim** as annual charges above 20,000 and ask a classic pricing question: *how does the frequency of large claims vary by age?*

Three steps: create an age band with `pd.cut`, flag the large claims, then group and average. Averaging a True/False column gives the proportion of Trues - which is exactly a frequency.

In [ ]:
bins   = [17, 29, 39, 49, 59, 64]
labels = ["18-29", "30-39", "40-49", "50-59", "60-64"]
insurance_data["age_band"] = pd.cut(insurance_data["age"], bins=bins, labels=labels)

insurance_data["large_claim"] = insurance_data["charges"] > 20000

frequency = (insurance_data
             .groupby("age_band", observed=True)
             .agg(lives=("age", "count"),
                  large_claims=("large_claim", "sum"),
                  frequency=("large_claim", "mean"),
                  avg_charges=("charges", "mean"))
             .round({"frequency": 3, "avg_charges": 0}))
frequency

**Reading the table.** Frequency rises steadily with age, as you would expect - but notice the jump in average charges is driven by a minority of large claims, not by everyone getting a little more expensive. Try changing the 20,000 threshold and see how stable the pattern is.

`groupby` is the pandas version of a pivot table, and `agg` lets you compute several summaries at once with names you choose. Here is the same idea by region and smoking status, which shows just how much of the cost is explained by smoking.

In [ ]:
by_region_smoker = insurance_data.pivot_table(values="charges", index="region", columns="smoker", aggfunc="mean").round(0)
by_region_smoker

## 6. Actuarial example: the severity distribution

Frequency tells you how often; **severity** tells you how much. A histogram of charges shows the long right tail that makes claims data awkward for the normal distribution and comfortable for the lognormal. Plotting the log of charges makes that visible.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(insurance_data["charges"], bins=40, edgecolor="black")
axes[0].set_title("Annual charges (severity)")
axes[0].set_xlabel("Charges")
axes[0].set_ylabel("Number of policyholders")

axes[1].hist(np.log(insurance_data["charges"]), bins=40, edgecolor="black")
axes[1].set_title("log(charges)")
axes[1].set_xlabel("log(Charges)")

plt.tight_layout()
plt.show()

**How a matplotlib chart is built:** create a figure, draw onto it (`hist`, `plot`, `scatter`...), label it, then `plt.show()`. Every chart you make follows that pattern. `figsize` is in inches - width first.

A scatter plot is the natural next question: does BMI explain charges? Colouring by smoker status is one extra argument and answers the question immediately.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=insurance_data, x="bmi", y="charges", hue="smoker", alpha=0.6)
plt.title("BMI vs annual charges, by smoking status")
plt.xlabel("BMI")
plt.ylabel("Annual charges")
plt.show()

## 7. Handling missing data

Our file is complete, so we will *create* some gaps to practise on: the cell below blanks out 10% of the BMI and charges values in a copy of the data. The two standard responses are to drop the incomplete rows or to fill the gaps with something sensible (here, the column mean). Neither is automatically right - the choice depends on why the data is missing, which is a judgement for you, not for the software.

In [ ]:
data_with_gaps = insurance_data.copy()
rng = np.random.default_rng(seed=42)                    # a seed makes the "random" gaps repeatable
gap_rows = rng.choice(data_with_gaps.index, size=134, replace=False)
data_with_gaps.loc[gap_rows, "bmi"] = np.nan
gap_rows = rng.choice(data_with_gaps.index, size=134, replace=False)
data_with_gaps.loc[gap_rows, "charges"] = np.nan

print("Missing values per column:")
print(data_with_gaps.isnull().sum())

In [ ]:
# Option 1: drop any row with a gap
dropped = data_with_gaps.dropna()
print("Rows after dropping:", len(dropped), "of", len(data_with_gaps))

# Option 2: fill gaps with the column mean
filled = data_with_gaps.copy()
filled["bmi"] = filled["bmi"].fillna(filled["bmi"].mean())
filled["charges"] = filled["charges"].fillna(filled["charges"].mean())
print("Missing after filling:", filled.isnull().sum().sum())

# Compare the effect on the average charge
print(f"Original mean charges: {insurance_data['charges'].mean():,.0f}")
print(f"After dropping:        {dropped['charges'].mean():,.0f}")
print(f"After mean-filling:    {filled['charges'].mean():,.0f}")

## 8. More charts with seaborn

seaborn understands DataFrames directly, so a chart that compares groups is usually one line. Box plots compare distributions; a heatmap of the correlation matrix is a quick way to see which variables move together.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=insurance_data, x="region", y="charges", hue="smoker")
plt.title("Distribution of charges by region and smoking status")
plt.show()

In [ ]:
# Correlation needs numbers, so turn the yes/no and male/female columns into 0/1 first
numeric = insurance_data[["age", "bmi", "children", "charges"]].copy()
numeric["smoker"] = (insurance_data["smoker"] == "yes").astype(int)
numeric["male"]   = (insurance_data["sex"] == "male").astype(int)

plt.figure(figsize=(7, 5))
sns.heatmap(numeric.corr(), annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation matrix")
plt.show()

**Reading the heatmap.** Smoking has a correlation of about 0.79 with charges; age and BMI matter far less on their own. This is the kind of chart that turns a data question into a pricing conversation in about ten seconds.

## 9. Getting results out: Excel and CSV

Most of your colleagues live in Excel, so the last step of an analysis is often to write a table out. pandas reads and writes Excel and CSV files in one line. In Colab the file appears in the folder pane on the left (click the folder icon, then the refresh icon), from where you can download it.

In [ ]:
frequency.to_excel("claim_frequency_by_age_band.xlsx")
by_region_smoker.to_csv("charges_by_region_smoker.csv")

# ...and reading it back in is just as easy
pd.read_excel("claim_frequency_by_age_band.xlsx")

## Your turn (5-10 minutes)

1. Build a frequency table like Section 5 but by **BMI band** (say under 25, 25-30, over 30) instead of age band.
2. Change the large-claim threshold to 30,000. Does the pattern by age still hold?
3. Draw a histogram of charges for smokers and non-smokers on the same axes (hint: call `plt.hist` twice with `alpha=0.5` and `label=` then `plt.legend()`).

In [ ]:
# Try your own code here

## What you have learned

You can load a file, inspect it, filter and sort it, group it into a summary table, handle missing values, draw the standard charts, and write the result to Excel. That workflow - load, explore, group, chart, export - is 80% of day-to-day actuarial data work, and it does not change when the file has five million rows instead of 1,338.

In the final notebook we hand this same dataset to an AI agent and let it do the pandas work for us.